**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/padilla-coreano-2022"], check=True)
    os.chdir("/content/encore/entries/padilla-coreano-2022")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Padilla-Coreano et al. (2022) from dandiset 001749

Paper. Padilla-Coreano N, Batra K, Patarino M, et al. (2022). Cortical ensembles orchestrate
social competition through hypothalamic outputs. *Nature* 603, 667-671.
[doi:10.1038/s41586-022-04507-5](https://doi.org/10.1038/s41586-022-04507-5). The claims file was
written from the PMC author manuscript (PMC9576144) with its Extended Data legends, and the Nature
supplementary information, which holds the Methods.

Dandiset. [DANDI:001749](https://dandiarchive.org/dandiset/001749/0.260311.1727), version
0.260311.1727. The paper cites no version of the dandiset, which was created in 2026, after the
paper. This is the only published version and the most recent one. The draft (modified
2026-09-23) lists the same 105 assets with the same paths, sizes and modification times, so the two
do not differ in content the analysis uses.

What the dandiset holds. 53 NWB files, one per implanted mouse and competition: 10 from the
"ephys" cohort (spikes and tone times only), 21 from the mPFC-LH ("chrimson") cohort and 22 from the
mPFC-BLA cohort (with AlphaTracker poses, port entries and, in 34 files, the nine hand-scored
behaviour labels of Figure 1f), and 52 behaviour videos (mp4). No file holds an alone session, a
phototagging light pulse, an optogenetic stimulation, the unimplanted cohort, the ex vivo recordings
or a body weight (file contents, in the data check).

Code and outside input. The HMM-GLM is ported from the paper's MATLAB code
(github.com/Tyelab/HMMGLM at commit 3fc3567815b9c891402635664a844320136d2fac) to `hmmglm.py`. The
paper's processed HMM-GLM input in that repository is read as a variant input and as a check of
the alignment; no outcome rests on it.

| Claim | Tested here |
|---|---|
| ed5b-rank1-wins-most | tested |
| ed5c-rewards-not-different | tested (rewards part; body weights are not in the dandiset) |
| ed5d-subordinates-slower-latency | tested |
| ed5e-port-occupancy-not-different | tested |
| ed5f-dominants-displace-more | tested |
| ed5g-time-displaced-not-different | tested |
| fig1i-hmmglm-decodes-behaviour, ed5j-six-states, ed5k-hmmglm-generalizes-across-rank | tested, on the 17 labelled files whose recording can be placed on the behaviour clock |
| fig1b, fig1c, ed1d, ed1g (unimplanted cohort) | blocked, data missing |
| fig1e-alphatracker-precision | blocked, data missing |
| ed7b-distance-by-trial-type-not-rank | blocked, metadata insufficient |
| Single-unit claims of Figures 2 and 3 and ED Figures 6 to 9 (22 claims, ed8fgh among them) | blocked, metadata insufficient |
| ed8ij, fig2e, ed9b (alone sessions) | blocked, data missing |
| fig4a, fig4b (2 claims), ed10c, fig4d, ed10ek (ex vivo, phototagging, optogenetics) | blocked, data missing |

That is 46 claims: 9 tested and 37 blocked. The reasons for each block are in the data check below
and in the attempt record.


In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")   # many small model fits (HMM-GLM); see hmmglm.py
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import time
import warnings
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from scipy import stats

import competition as c
import hmmglm as hg

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
files = c.load_all()
print(f"loaded {len(files)} files from dandiset {c.DANDISET_ID} version {c.DANDISET_VERSION} in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
info = pd.DataFrame([{
    "file": r["path"].split("/")[0][4:], "cohort": r["cohort"], "mouse": r["mouse"], "competitor": r["competitor"],
    "rank": r["rank"], "other_rank": r["other_rank"], "start": r["session_start"][:16], "units": r["n_units"],
    "rec_s": round(r["es_shape"][0] / r["es_rate"]), "last_spike_s": round(r["last_spike"]),
    "cues": len(r["cs"]), "last_cue_s": round(r["cs"][-1]), "deliveries": len(r["us"]),
    "port_entries": None if r["pe_start"] is None else len(r["pe_start"]),
    "label_trials": None if r["labels"] is None else r["labels"].shape[0],
    "pair_in_dataset": r["pair_in_dataset"]} for r in files])
print(info.to_string(index=False))
print("\nFiles per cohort:", info.cohort.value_counts().to_dict())
print("Mice:", info.mouse.nunique(), "; per cohort:", info.groupby("cohort").mouse.nunique().to_dict())
sessions = {c.session_key(r) for r in files}
print("Competition sessions (date in file name, cage, pair):", len(sessions))
both = [k for k in sessions if sum(c.session_key(r) == k for r in files) == 2]
print("Sessions with the files of both mice:", len(both))
print("Files whose pair_in_dataset is 'not included':", (info.pair_in_dataset == "not included").sum(), "of", len(info))
print("Recording channels and rate:", sorted({(r["es_shape"][1], r["es_rate"], r["es_unit"]) for r in files}))

The 53 files hold 20 mice: 5 in the "ephys" cohort, 7 mPFC-LH and 8 mPFC-BLA. The paper's mPFC
recordings come from 20 mice with cells. Every file says `pair_in_dataset: not included`, but for
the sessions counted above both mice of the pair are in the dandiset; the field is wrong for those
files. The session count uses the date in the file name. Some file names and session start times
disagree; see the next cell.

What the files hold. The next cell lists every group and dataset of one file per cohort, with the
columns of each table, and the subject record of every file.

In [ ]:
al = c.assets()
shown = {}
for coh in ("ephys", "chrimson", "BLA"):
    a = next(x for x in al if coh in x["path"].split("/")[0])
    t = c.tree(a)
    poses = [k for k in t if k.startswith("processing/behavior_Mouse")]
    print(f"--- {a['path'].split('/')[0]}")
    for k, v in t.items():
        if k not in poses:
            print(f"  {k}: {v}")
    if poses:
        parts = sorted({k.split("/")[3] for k in poses})
        print(f"  processing/behavior_Mouse1 and behavior_Mouse2: PoseEstimation series {parts}, each with data, timestamps and reference_frame")
    shown[coh] = t
names = set().union(*[set(t) for t in shown.values()])
for word in ("video_gap", "weight", "stimulus", "opto", "light", "laser", "alone", "epochs", "trials", "analysis"):
    print(f"Datasets or tables whose path contains {word!r}:", sorted(k for k in names if word in k.lower()))
print("\nSubject fields (all 53 files):", sorted({k for r in files for k in r["subject"]}))
print("Values of the fields other than the description:")
for k in sorted({k for r in files for k in r["subject"]} - {"description", "subject_id"}):
    print(f"  {k}:", sorted({str(r['subject'][k]) for r in files if k in r['subject']}))
print("Files whose subject description mentions a weight:", sum("weigh" in r["description"].lower() or " g " in r["description"] for r in files))
print("Example description:", files[0]["description"])
print("Subject descriptions (all 53 files) containing each word:",
      {w: sum(w in r["description"].lower() for r in files) for w in ("compet", "cagemate", "alone")},
      "; distinct descriptions:", len({r["description"] for r in files}))
print("\nUnits table columns (all 53 files):", sorted({c_ for t in shown.values() for k, v in t.items() if k == "units (columns)" for c_ in v}))
print("Quality labels (all 53 files):", sorted({x for r in files for x in r["quality"]}))
from dandi.dandiapi import DandiAPIClient
other = [x.path for x in DandiAPIClient().get_dandiset(c.DANDISET_ID, c.DANDISET_VERSION).get_assets() if not x.path.endswith(".nwb")]
print("\nAssets other than NWB files:", len(other), "; extensions:", sorted({p.rsplit(".", 1)[-1] for p in other}))
print("Video description of one file:", files[0]["video"][:300])

The files hold the recording, the Klusta clusters, the tone and delivery times and, in the
mPFC-LH and mPFC-BLA files, port entries, poses and labels. There is no `video_gap`, no stimulus or
optogenetic series, no light-pulse or alone epoch, and no table of trials. The subject record has
no weight field, and no subject description gives one, so body weights (ED Fig. 5c) are not in the
dandiset. The units table has a `quality` column and the cluster id only (see "Units" below). The 52
other assets are the behaviour videos (mp4). The dandiset description says the LED in the video is
on during the tone so that the neural and behavioural data can be aligned. That LED places the tone
in the video, and the labels and port entries are already on the tone clock (below); it does not
give the offset between the video and the logger's recording, which is the `video_gap` that is
missing. The videos are not read here.

In [ ]:
# Ranks: one rank per mouse across files, and the competitor's rank as its own file gives it.
ranks = {}
conflicts = []
for r in files:
    for m, k in ((r["mouse"], r["rank"]), (r["competitor"], r["other_rank"])):
        if ranks.setdefault(m, k) != k:
            conflicts.append((m, ranks[m], k))
print("Mice with more than one rank across files:", conflicts)
print("Rank per mouse:", dict(sorted(ranks.items())))
mismatch = [(r["path"].split("/")[0], r["description"][:95]) for r in files
            if (r["desc_animal"], r["desc_other"]) != (r["animal"], r["other_animal"])]
print("\nFiles whose subject description names other animals than the file name:")
for m in mismatch:
    print("  ", m)
dates = [(r["path"].split("/")[0], r["session_start"]) for r in files if r["session_start"][:10].replace("-", "") != r["date"]]
print("\nFiles whose session start date differs from the date in the file name:", dates)
print("\nCompetitions per own rank (53 files):", info["rank"].value_counts().sort_index().to_dict())

Every mouse has one rank across files, and the competitor's rank in a file agrees with the
competitor's own files. In one file the subject description names mouse 2 against mouse 3 while the
file name says mouse 3 against mouse 2; the ranks it gives (4 for the subject, 1 for the competitor)
are those of mice 3 and 2 elsewhere, so the description swaps the animal numbers but keeps the
ranks of the file's subject. One file named 20200121 has a session start of 2020-01-22 13:30. Its
labels mirror those of the file of its competitor in the 2020-01-21 13:15 session (shown below), so
the start time is wrong.

In [ ]:
schedules = {tuple(np.round(r["cs"], 2)) for r in files}
shared = 0
while all(len(x) > shared for x in schedules) and len({x[shared] for x in schedules}) == 1:
    shared += 1
print("Distinct tone schedules:", len(schedules), "; leading tone onsets shared by all 53 files:", shared)
for cohort_is_ephys in (True, False):
    sub = {tuple(np.round(r["cs"], 2)) for r in files if (r["cohort"] == "mPFC") == cohort_is_ephys}
    longest = max(sub, key=len)
    print(f"  {'ephys' if cohort_is_ephys else 'mPFC-LH and mPFC-BLA'} files: {len(sub)} schedules, all prefixes of the longest ({len(longest)} tones):",
          all(x == longest[:len(x)] for x in sub), "; ITIs (s)", np.diff(longest).min(), "to", np.diff(longest).max())
print("Tone durations (s):", sorted({round(float(d), 3) for r in files for d in r["cs_stop"] - r["cs"]}))
d = np.concatenate([r["us"] - r["cs"][np.searchsorted(r["cs"], r["us"]) - 1] for r in files])
print("Delivery minus the tone onset before it (s):", sorted(set(np.round(d, 3))), "; the Methods give 2 s")
print("Files with fewer deliveries than tones:", [(r["path"][4:28], len(r["cs"]), len(r["us"])) for r in files if len(r["us"]) != len(r["cs"])])
print("Files whose last tone is after the end of the recording:",
      int((info.last_cue_s > info.rec_s).sum()), "of", len(info), info.loc[info.last_cue_s > info.rec_s, "cohort"].value_counts().to_dict())
lab = info.dropna(subset=["label_trials"])
print("Labelled files with fewer label trials than tones:", int((lab.label_trials < lab.cues).sum()), "of", len(lab))

All 53 files share their first 17 tone onsets (60.01, 140.01, 230.01 s and so on); after them the
"ephys" files follow one schedule and the other files another, each cut at different lengths, so the
tone times are the program's schedule, not times recorded in each session. The port entries and labels agree with it
(below), so for the files with port entries it is the schedule that was run, in the behaviour clock.
The deliveries are 0.49 s after tone onset in every file, where the Methods give 2 s; the analysis
below takes 2 s from the Methods and shows 0.49 s as a variant. In 7 of the 10 "ephys" files the
last tones fall after the end of the neural recording. Five files have fewer deliveries than tones. The labels hold fewer trials than tones in some files;
which tones they belong to is checked below against the port entries.

In [ ]:
pe_files = [r for r in files if r["labels"] is not None and r["pe_start"] is not None]
rows = []
for r in pe_files:
    selfin = np.isin(r["labels"], c.SELF_IN).astype(float)
    n = min(len(r["cs"]), selfin.shape[0])
    def corr(lag):
        occ = c.port_occupancy_on_label_grid(r, lag)
        return np.corrcoef(selfin[:n].ravel(), occ[:n].ravel())[0, 1]
    occ = c.port_occupancy_on_label_grid(r, 0.0)
    shifted = np.corrcoef(selfin[1:n].ravel(), occ[:n - 1].ravel())[0, 1]   # labels of trial j+1 against entries of trial j
    scan = {lag: corr(lag) for lag in np.arange(-3, 3.01, 0.25)}
    best = max(scan, key=scan.get)
    rows.append({"file": r["path"][4:28], "r_lag0": corr(0.0), "best_lag_s": best, "r_best": scan[best], "r_next_trial": shifted})
pl = pd.DataFrame(rows)
print(pl.round(3).to_string(index=False))
print("\nMedian r at lag 0:", round(pl.r_lag0.median(), 3), "; median r with labels of the next trial:", round(pl.r_next_trial.median(), 3),
      "; files whose best lag within +-3 s is within 0.5 s of 0:", int((pl.best_lag_s.abs() <= 0.5).sum()), "of", len(pl))

The subject's port entries (a lickometer or beam signal) and the hand-scored "self in port" labels
agree at lag 0 (correlation over the label bins of each trial; the contrast is the same correlation
with the labels of the next trial). This shows that the tone schedule, the port entries and the
labels share one clock, the behaviour clock, in the files with both. The files whose correlation is low are named in the table;
the next cell asks, for the files with fewer label trials than tones, which tones the label trials are.

In [ ]:
# Which tones the label trials belong to. Label trial j is set against tone j + k, for every k the
# number of tones allows, and the "self in port" labels are correlated with the port entries.
rows = []
for r in pe_files:
    sc = c.tone_shift_scan(r)
    k = int(np.nanargmax(sc))
    rows.append({"file": r["path"].split("/")[0][4:], "label_trials": r["labels"].shape[0], "tones": len(r["cs"]),
                 "r_k0": sc[0], "best_k": k, "r_best": sc[k], "r_each_k": " ".join(f"{x:.2f}" for x in sc) if len(sc) > 1 else ""})
shift = pd.DataFrame(rows)
print(shift.round(3).to_string(index=False))
fewer = shift[shift.label_trials < shift.tones]
print("\nFiles with fewer label trials than tones:", len(fewer), "; of them best at k = 0:", int((fewer.best_k == 0).sum()),
      "; best at k > 0 by more than 0.1:", int(((fewer.best_k > 0) & (fewer.r_best - fewer.r_k0 > 0.1)).sum()))
print("Files with r below 0.3 at k = 0:", shift[shift.r_k0 < 0.3].file.tolist())
print("Files with r below 0.3 at their best k:", shift[shift.r_best < 0.3].file.tolist())

Where a file has as many label trials as tones, label trial *j* is tone *j*. Of the files with
fewer label trials, several match the port entries better when the label trials start at a later
tone (k above 0, listed): in those files the first tones were not scored, and with k = 0 the
labels of one trial are set against the entries of another. In others no k gives a clear match.
The latency test of ED Fig. 5d, which sets labels against port entries, is shown both with k = 0
and with each file's best k, and with the files below r = 0.3 left out. The HMM-GLM input below
uses each file's best k.

In [ ]:
al = c.assets()
started = time.time()
with ThreadPoolExecutor(8) as ex:
    clocks = pd.DataFrame(list(ex.map(c.clock_checks, al)))
clocks["file"] = clocks.path.str.split("/").str[0].str[4:]
print(f"clock checks of {len(clocks)} files in {time.time() - started:.0f} s")
cols = ["file", "raw_peak_sd_1.0", "raw_lag_1.0", "raw_peak_sd_1.5", "raw_lag_1.5"]
print(clocks[cols].round(3).to_string(index=False))
print("\nPeak of the spike/raw cross-correlation (SD over lags), median over files: stored times",
      round(clocks["raw_peak_sd_1.0"].median(), 1), "; times x 1.5", round(clocks["raw_peak_sd_1.5"].median(), 1))
print("Files with a peak above 10 SD: stored", int((clocks["raw_peak_sd_1.0"] > 10).sum()), "; x 1.5", int((clocks["raw_peak_sd_1.5"] > 10).sum()))
print("Last spike / recording length, range over files:", round((info.last_spike_s / info.rec_s).min(), 3), "to", round((info.last_spike_s / info.rec_s).max(), 3))
off = clocks[(clocks["raw_peak_sd_1.5"] > 10) & (clocks["raw_lag_1.5"].abs() > 0.1)]
print("Files that line up only after an offset of more than 0.1 s (x 1.5):", len(off), "; offsets (s) from", round(off["raw_lag_1.5"].min(), 2), "to", round(off["raw_lag_1.5"].max(), 2))
print("Files without a peak above 10 SD at either scale:", clocks.loc[(clocks["raw_peak_sd_1.5"] <= 10) & (clocks["raw_peak_sd_1.0"] <= 10), "file"].tolist())

The stored spike times do not line up with the raw recording: their cross-correlation with threshold
crossings of 20 s of the band-passed `ElectricalSeries` (offsets of -60 to +60 s) has no peak above
the 3 to 7 SD of chance. The same times multiplied by 1.5 line up, at a lag of a few milliseconds in
most files, and in every file the last spike falls at about two thirds of the recording length. The
spike times were divided by 30 kHz where the logger samples at 20 kHz (as the file states). This is
a format error; times x 1.5 are used below. In the files listed above as lining up only after an
offset, the scaled spikes match the recording shifted by seconds, so in those files the sorted
spikes and the raw recording do not share a start either.

In [ ]:
pe = clocks.dropna(subset=["pe_lag"]).copy()
pe["clear"] = pe.pe_peak > 2 * pe.pe_p99
cols = ["file", "n_entries", "pe_lag", "pe_peak", "pe_p99", "pe_median", "pe_at_zero", "cs_at_pe_lag", "cs_p99", "clear"]
print(pe[cols].round(2).to_string(index=False))
print("\nFiles with port entries:", len(pe), "; with a clear peak (more than twice the 99th percentile over offsets):", int(pe.clear.sum()))
print("Offsets of the clear peaks (s): median", round(pe[pe.clear].pe_lag.median(), 1), "range", round(pe[pe.clear].pe_lag.min(), 1), "to", round(pe[pe.clear].pe_lag.max(), 1))
print("Score at offset 0 against the 99th percentile:", int((pe.pe_at_zero > pe.pe_p99).sum()), "files above")
by_day = pe[pe.clear].assign(day=pe.file.str[:8] + "-" + pe.file.str.extract(r"-(\w+\d)-")[0]).groupby("day").pe_lag.agg(["count", "min", "max"])
print("\nClear offsets by day and cage:\n", by_day.round(1).to_string())

In [ ]:
# Recordings deposited twice: the two files of a session share the video and poses; compare their
# recordings, sample counts and units, and look for equal sample counts across all files.
url = {a["path"]: a["url"] for a in al}
rows = []
for k in sorted(lab_key for lab_key in {c.session_key(r) for r in files}):
    v = [r for r in files if c.session_key(r) == k]
    if len(v) != 2:
        continue
    fa, fb = c.open_nwb(url[v[0]["path"]]), c.open_nwb(url[v[1]["path"]])
    xa = fa["acquisition/ElectricalSeries/data"][6_000_000:6_002_000, :]
    xb = fb["acquisition/ElectricalSeries/data"][6_000_000:6_002_000, :]
    rows.append({"session": "-".join(k), "samples_a": v[0]["es_shape"][0], "samples_b": v[1]["es_shape"][0],
                 "same_block": bool(np.array_equal(xa, xb)), "units_a": v[0]["n_units"], "units_b": v[1]["n_units"],
                 "same_spike_counts": bool(np.array_equal(np.sort(v[0]["spike_counts"]), np.sort(v[1]["spike_counts"])))})
dup = pd.DataFrame(rows)
print(dup.to_string(index=False))
print("\nPairs of files of one session with identical recordings or spike counts:", int((dup.same_block | dup.same_spike_counts).sum()))

import hashlib
def fingerprint(a):
    d = c.open_nwb(a["url"])["acquisition/ElectricalSeries/data"]
    return a["path"].split("/")[0][4:], d.shape[0], hashlib.md5(d[2_000_000:2_000_500, :].tobytes()).hexdigest()
with ThreadPoolExecutor(12) as ex:
    fps = pd.DataFrame(list(ex.map(fingerprint, al)), columns=["file", "samples", "md5_of_500_samples"])
groups = fps[fps.duplicated("md5_of_500_samples", keep=False)]
print("Files whose recordings share a block of 500 samples at 100 s (all 53 files):")
print(groups.to_string(index=False))
same = [r for r in files if r["path"].split("/")[0][4:] in set(groups.file)]
if len(same) == 2:
    f0, f1 = c.open_nwb(url[same[0]["path"]]), c.open_nwb(url[same[1]["path"]])
    print("Identical at samples 1e6, 3e7 and 6e7:", [bool(np.array_equal(f0["acquisition/ElectricalSeries/data"][o:o + 2000, :], f1["acquisition/ElectricalSeries/data"][o:o + 2000, :])) for o in (1_000_000, 30_000_000, 60_000_000)],
          "; units", [r["n_units"] for r in same], "; session starts", [r["session_start"][:16] for r in same])

Within each session the two loggers hold different signals, lengths and clusters; the two files of
a session share the behaviour video and its poses, and their labels are two views of one scoring
(below). Across all files, one recording is deposited twice: the `ElectricalSeries` of
sub-20191025-chrimson2-3-1v3 and sub-20191029-chrimson2-3-1v3 (the same mouse against the same
competitor, four days apart) are the same signal. Their units differ, and the 20191025 file's units
match its recording at no offset (the spike check above), so the 20191025 file most likely carries
the 20191029 recording in place of its own. Neither file is labelled, so no behavioural test below
uses them.

The spikes (x 1.5) and the behaviour clock are offset. Placing the port entries on the spike clock
at each offset from -300 to +300 s, the units respond to them at a single offset, different in each
session (from a few seconds to tens of seconds), and not at offset 0. Files recorded on the same day
in the same cage have nearly the same offset; the tones give a weaker response at the same offset.
The dandiset description says this offset is in a `video_gap` variable of the behavioural events;
no file holds such a variable. For the "ephys" files, which have no port entries, it cannot be
estimated this way, and for the files without a clear peak it cannot be estimated at all.

In [ ]:
q = np.concatenate([r["spike_counts"] / (r["es_shape"][0] / r["es_rate"]) for r in files])
print("Clusters:", len(q), "; quality labels:", sorted({x for r in files for x in r["quality"]}))
print("Firing rate (Hz) percentiles 5, 25, 50, 75, 95:", np.percentile(q, [5, 25, 50, 75, 95]).round(2))
print("Clusters at or above 0.05 Hz:", int((q >= 0.05).sum()), "; above 1 Hz:", int((q > 1).sum()), "; above 100 Hz:", int((q > 100).sum()))
dom = sum(r["n_units"] for r in files if r["rank"] < r["other_rank"]); sub = sum(r["n_units"] for r in files if r["rank"] > r["other_rank"])
print("Clusters in files of the relative dominant:", dom, "; of the relative subordinate:", sub)
print("Paper: 998 cells (507 dom, 490 sub in Figure 2a), 913 kept for clustering")

The `units` tables hold every Klusta cluster, 5,879 in all, each labelled `unsorted`, where the
paper analyses 998 manually curated cells. The table has the column for a curation label (good, mua,
noise) but no cluster carries one, and no simple rule on firing rate comes near 998. The paper's
cells cannot be told apart from multi-unit and noise clusters, so every claim on single units is
blocked as `metadata_insufficient`.

In [ ]:
lf = [r for r in files if r["labels"] is not None]
print("Labelled files:", len(lf), "; mice:", len({r["mouse"] for r in lf}), "; sessions:", len({c.session_key(r) for r in lf}),
      "; trials:", sum(r["labels"].shape[0] for r in lf))
print("Label bins per trial:", sorted({r["labels"].shape[1] for r in lf}), "at", sorted({r["label_rate"] for r in lf}), "Hz")
print("Label trials in files of the relative dominant:", sum(r["labels"].shape[0] for r in lf if r["rank"] < r["other_rank"]),
      "; of the relative subordinate:", sum(r["labels"].shape[0] for r in lf if r["rank"] > r["other_rank"]))
labelled_names = {r["path"].split("/")[0][4:] for r in lf}
print("Labelled files with a clear offset between spikes and port entries (data check above):",
      len(labelled_names & set(pe[pe.clear].file)), "of", len(labelled_names))
allv = np.concatenate([r["labels"].ravel() for r in lf])
print("Label codes and bins:", dict(zip(*np.unique(allv, return_counts=True))))
# Pairs: how the two views of one session relate.
by = {}
for r in lf:
    by.setdefault(c.session_key(r), []).append(r)
rows, co = [], {}
for k, v in sorted(by.items()):
    if len(v) == 2 and v[0]["labels"].shape == v[1]["labels"].shape:
        A, B = v[0]["labels"], v[1]["labels"]
        pos = np.isin(A, list(c.MIRROR_POSITION))
        agree = np.mean(np.vectorize(lambda x: c.MIRROR_POSITION.get(x, -1))(A[pos]) == B[pos])
        rows.append({"session": "-".join(k), "position bins agreeing": round(float(agree), 3)})
        for a, b in zip(A.ravel(), B.ravel()):
            if a in (1, 2, 3) or b in (1, 2, 3):
                co[(int(a), int(b))] = co.get((int(a), int(b)), 0) + 1
print(pd.DataFrame(rows).to_string(index=False))
print("\nPush, resist and displaced bins in one view (first) against the other view (second):", dict(sorted(co.items())))

The 34 labelled files hold 965 trials from 14 mice, 280 bins at 20 Hz each (-2 to +12 s from tone
onset, as the description says), which matches the 965 trials of the paper's HMM-GLM dataset
(Methods 17; 13 mice, 32 sessions). Where both mice of a session are labelled, one file's labels are
the other's seen from the competitor (self in port against other in port, and so on); most pairs
agree in nearly every bin, two in only 67.5% and 75.6%. A push (1) in one view is displaced (3) or
resist (2) in the other, and resist and displaced are pushes in the other view, so one file shows the
outcome of the competitor's pushes but not of its subject's own. Labels change in whole seconds (runs of 20 bins).

In [ ]:
with ThreadPoolExecutor(8) as ex:
    pose = pd.DataFrame(list(ex.map(c.pose_summary, [a for a in al if "ephys" not in a["path"].split("/")[0]])))
pose["file"] = pose.path.str.split("/").str[0].str[4:]
print(pose.drop(columns=["path"]).round(3).to_string(index=False))
print("\nFiles whose poses are all missing (both mice):", int(((pose.Mouse1_nan == 1) & (pose.Mouse2_nan == 1)).sum()), "of", len(pose))
print("Body parts:", pose.parts.unique())

AlphaTracker poses are given for two mice, `Mouse1` and `Mouse2`, with ears, tail base and logger
(no nose, which the Methods say was tracked). Nothing says which of them is the file's subject, and
the files of many sessions hold no position at all (all values missing). The port location the paper
marked by hand is not in the files. Distance to the port (ED Fig. 7b) cannot be computed.

## Trial outcomes

The claims below use the trial outcomes of the labelled files, from the Methods ("Reward
competition behavioral analysis", quoted in `competition.py`). Relative dominant (dom) is the
mouse of the pair with the higher tube-test rank (the smaller number). The two readings of a
"complete switch" give the same outcomes here because the labels change only on whole seconds, so
the 1.5 to 3 s window holds at most one change; the variant tests nothing beyond the main rule.

In [ ]:
def outcomes(r, delivery_s=c.DELIVERY_S, switch_rule="first_last"):
    return c.trial_outcomes(r["labels"], delivery_s, switch_rule)

rows = []
for r in lf:
    for name, kw in {"main": {}, "delivery 0.49 s": {"delivery_s": 0.49}, "switch = both in port": {"switch_rule": "any"}}.items():
        o = outcomes(r, **kw)
        rows.append({"reading": name, "file": r["path"][4:28], "win": int((o == "win").sum()), "lose": int((o == "lose").sum()), "excluded": int((o == "excluded").sum())})
oc = pd.DataFrame(rows)
print(oc.groupby("reading")[["win", "lose", "excluded"]].sum().to_string())

### ed5b-rank1-wins-most

> In the recording cohort, rank 1 mice won the highest percentage of competitions. (ED Fig. 5b;
> expected 12, 12, 15 and 14 competitions for ranks 1 to 4.)

A competition is won by the mouse that won more of its trials. A file's competition is decided from
its own labels or, when it has none, from the labels of its competitor's file (whose losses are its
wins).

In [ ]:
lab_by_session = {}
for r in lf:
    lab_by_session.setdefault(c.session_key(r), []).append(r)

def competition_result(r, **kw):
    if r["labels"] is not None:
        o = outcomes(r, **kw); w, l = (o == "win").sum(), (o == "lose").sum()
        src = "own labels"
    else:
        others = [x for x in lab_by_session.get(c.session_key(r), []) if x["mouse"] == r["competitor"]]
        if not others:
            return None, "no labels"
        o = outcomes(others[0], **kw); w, l = (o == "lose").sum(), (o == "win").sum()
        src = "competitor's labels"
    return ("won" if w > l else "lost" if l > w else "tie"), src

for name, kw in {"main": {}, "delivery 0.49 s": {"delivery_s": 0.49}, "switch = both in port": {"switch_rule": "any"}}.items():
    res = pd.DataFrame([{"file": r["path"][4:28], "rank": r["rank"], "result": competition_result(r, **kw)[0], "source": competition_result(r, **kw)[1]} for r in files])
    known = res.dropna(subset=["result"])
    t = known.groupby("rank").result.value_counts().unstack(fill_value=0)
    for col in ("won", "lost", "tie"):
        if col not in t:
            t[col] = 0
    t["competitions (all files)"] = res.groupby("rank").size()
    t["% won, tie as not won"] = (100 * t.won / (t.won + t.lost + t.tie)).round(1)
    t["% won, ties left out"] = (100 * t.won / (t.won + t.lost)).round(1)
    print(f"--- {name}")
    if name == "main":
        print("Sources:", res.source.value_counts().to_dict())
    print(t.to_string())
    for col in ("% won, tie as not won", "% won, ties left out"):
        print(f"  highest {col}: rank {int(t[col].idxmax())}")

**Result: reproduced.** Rank 1 mice won 10 of the 11 competitions whose outcome the labels give
(90.9%), against 12.5% for rank 2, 44.4% for rank 3 and 62.5% for rank 4, under both readings of a tie
(there are none) and under the variants of the trial rule. The competitions per rank over the 53
files (12, 11, 15, 15) are within tolerance of the expected 12, 12, 15 and 14. Only 36 of the 53
competitions have labels, of the file's subject or its competitor.

### ed5c-rewards-not-different

> Among mice wearing loggers, relative dominants and subordinates obtained similar numbers of
> rewards, while the body weight difference between competitors correlated with rewards won.

Rewards are trials won. Readings of the pairs: (main) each session once, from the dominant's file
when there is one; (a) each labelled file, with its subject and its competitor. Reading (a) counts
the two files of a session with both files as two pairs, though they are two views of one session,
so its n is not independent; it is shown because the paper's 22 pairs and its other "session"
counts are closer to files than to sessions. Body weights are not in the dandiset (file contents,
above), so the correlation is not tested.


In [ ]:
def per_file(fn):
    rows = []
    for r in lf:
        s, o = fn(r)
        dom_self = r["rank"] < r["other_rank"]
        rows.append({"file": r["path"][4:28], "session": c.session_key(r), "dom_file": dom_self,
                     "dom": s if dom_self else o, "sub": o if dom_self else s,
                     "dom_mouse": r["mouse"] if dom_self else r["competitor"], "sub_mouse": r["competitor"] if dom_self else r["mouse"]})
    return pd.DataFrame(rows)

def per_session(d):
    return d.sort_values("dom_file", ascending=False).groupby("session").first().reset_index()

def paired(d, label):
    d = d.dropna(subset=["dom", "sub"])
    t = stats.ttest_rel(d["dom"], d["sub"])
    diff = d["dom"] - d["sub"]
    print(f"{label}: n = {len(d)}, dom mean {d['dom'].mean():.2f}, sub mean {d['sub'].mean():.2f}, "
          f"mean difference {diff.mean():.2f} (95% CI {diff.mean() - 1.96 * diff.std(ddof=1) / np.sqrt(len(d)):.2f} to "
          f"{diff.mean() + 1.96 * diff.std(ddof=1) / np.sqrt(len(d)):.2f}), paired t = {t.statistic:.2f}, p = {t.pvalue:.3g}")
    return t.pvalue

def rewards(r, **kw):
    o = outcomes(r, **kw)
    return (o == "win").sum(), (o == "lose").sum()

rw = per_file(rewards)
p_b = paired(per_session(rw), "(main) sessions")
p_a = paired(rw, "(a) files")
paired(per_session(per_file(lambda r: rewards(r, delivery_s=0.49))), "sessions, delivery at 0.49 s")
paired(per_session(per_file(lambda r: rewards(r, switch_rule="any"))), "sessions, switch = both in port")
print("\nRewards by dominant mouse (reading a):")
print(rw.groupby("dom_mouse")[["dom", "sub"]].agg(["count", "sum"]).to_string())

**Result: partially reproduced.** The rewards part holds: dominants and subordinates did not differ
in rewards by the paired t-test, 14.35 against 12.15 rewards over 20 sessions (main; p = 0.498),
14.12 against 12.38 over the 34 files (p = 0.505), and p = 0.543 with delivery at 0.49 s. The
correlation of body weight difference with rewards won could not be tested: no file holds a body
weight (file contents, above).

### ed5d-subordinates-slower-latency

> In win trials, relative subordinates had longer latencies to pick up the reward than relative
> dominants. (Wilcoxon rank-sum p = 0.012; 326 dom and 358 sub trials.)

Each file's subject in its own win trials, from its port entries (the criterion's "latency of the
winner's port entry"): seconds from tone onset to the first port entry, 0 when the mouse is in the
port at onset (reading a; reading b, from delivery at 2 s, differs by a constant and gives the same
tests). How a mouse already in the port is counted is not given; the variant takes the first entry
that starts after onset. A second variant takes the latency from the hand-scored labels (first bin
labelled self in port at or after onset), over all win trials and over the trials of the main
test. Since the main test sets each label trial against a tone, it is also run with each file's
best tone shift k (data check) and without the files whose labels and port entries agree at r
below 0.3. The cut-off of 0.3 is a choice; the same tests are shown at cut-offs from 0 to 0.5.

In [ ]:
def entry_latency(r, j, in_port_is_zero=True):
    t = r["cs"][j]
    s, e = r["pe_start"], r["pe_stop"]
    if in_port_is_zero and np.any((s <= t) & (e > t)):
        return 0.0
    k = s[s >= t]
    return float(k[0] - t) if len(k) and k[0] - t <= 12 else np.nan

sh = shift.set_index("file")
rows = []
for r in lf:
    o = outcomes(r)
    lat = c.latency_to_port(r["labels"], "self")
    name = r["path"].split("/")[0][4:]
    has = r["pe_start"] is not None
    k = int(sh.loc[name, "best_k"]) if has else 0
    for j in np.flatnonzero(o == "win"):
        rows.append({"mouse": r["mouse"], "file": name, "group": "dom" if r["rank"] < r["other_rank"] else "sub",
                     "r_k0": sh.loc[name, "r_k0"] if has else np.nan, "r_best": sh.loc[name, "r_best"] if has else np.nan,
                     "latency_labels": lat[j],
                     "latency_entries": entry_latency(r, j) if has and j < len(r["cs"]) else np.nan,
                     "latency_next_entry": entry_latency(r, j, False) if has and j < len(r["cs"]) else np.nan,
                     "latency_entries_shift": entry_latency(r, j + k) if has and j + k < len(r["cs"]) else np.nan})
lt = pd.DataFrame(rows)

def latency_test(d, col, label):
    d = d.dropna(subset=[col])
    a, b = d[d.group == "dom"][col], d[d.group == "sub"][col]
    rs = stats.mannwhitneyu(b, a, alternative="two-sided")
    ks = stats.ks_2samp(a, b)
    print(f"{label}: dom n = {len(a)}, median {a.median():.2f} s, mean {a.mean():.2f}; sub n = {len(b)}, median {b.median():.2f} s, mean {b.mean():.2f}; "
          f"rank-sum p = {rs.pvalue:.3g} (common-language effect sub > dom {rs.statistic / (len(a) * len(b)):.2f}); KS p = {ks.pvalue:.3g}")
    return rs.pvalue

p_main = latency_test(lt, "latency_entries", "(main) port entries, tone j, in port at onset = 0")
latency_test(lt, "latency_next_entry", "port entries, tone j, next entry after onset")
latency_test(lt, "latency_labels", "labels, all win trials")
latency_test(lt.dropna(subset=["latency_entries"]), "latency_labels", "labels, the trials of the main test")
latency_test(lt[lt.r_k0 >= 0.3], "latency_entries", "port entries, tone j, files with r >= 0.3 at k = 0")
latency_test(lt, "latency_entries_shift", "port entries, tone j + best k")
latency_test(lt[lt.r_best >= 0.3], "latency_entries_shift", "port entries, tone j + best k, files with r >= 0.3 at best k")
print("Files left out at r < 0.3 or without a value at k = 0:", sorted(lt[~(lt.r_k0 >= 0.3)].file.dropna().unique()))
print("Files left out at r < 0.3 at best k:", sorted(lt[~(lt.r_best >= 0.3)].file.dropna().unique()))
# The cut-off of r >= 0.3 is a choice: the same tests at other cut-offs.
cut_rows = []
for cut in (0.0, 0.1, 0.2, 0.3, 0.4, 0.5):
    for col, rcol, name in (("latency_entries", "r_k0", "tone j"), ("latency_entries_shift", "r_best", "tone j + best k")):
        d = lt[lt[rcol] >= cut].dropna(subset=[col])
        a, b = d[d.group == "dom"][col], d[d.group == "sub"][col]
        cut_rows.append({"r at least": cut, "tones": name, "files": d.file.nunique(), "dom n": len(a), "sub n": len(b),
                         "dom median": a.median(), "sub median": b.median(), "rank-sum p": stats.mannwhitneyu(b, a).pvalue,
                         "KS p": stats.ks_2samp(a, b).pvalue})
print("\nLatency from the port entries by cut-off on the agreement r of labels and port entries:")
print(pd.DataFrame(cut_rows).round(4).to_string(index=False))
print("\nRange (s): entries", lt.latency_entries.min(), "to", round(lt.latency_entries.max(), 2), "; labels", lt.latency_labels.min(), "to", lt.latency_labels.max())
print("Win trials without a port entry within 12 s, or in a file without port entries:", int(lt.latency_entries.isna().sum()), "of", len(lt))
print("\nPer mouse and group:")
print(lt.groupby(["group", "mouse"])[["latency_entries", "latency_labels"]].agg(["count", "median"]).round(2).to_string())
for col in ("latency_entries", "latency_labels"):
    mm = lt.groupby(["group", "mouse"])[col].mean().reset_index().dropna()
    print(f"Mouse means of {col}, sub against dom (rank-sum over {len(mm)} mouse groups): p =",
          round(stats.mannwhitneyu(mm[mm.group == "sub"][col], mm[mm.group == "dom"][col]).pvalue, 3))

**Result: not reproduced.** Setting every label trial against the tone of the same index, subordinates
entered the port later than dominants in their win trials (median 1.47 against 1.01 s over 189 and
211 trials, rank-sum p = 0.0196, KS p = 0.0036; next entry for a mouse already in the port, p =
0.0204). This keeps files whose labels and port entries do not agree, so that a win trial is set
against the entries of another trial. With each file's best tone shift p = 0.0458 over all files;
without the files whose labels and entries agree at r below a cut-off, p is 0.0623, 0.375, 0.375
and 0.0662 at cut-offs of 0.2, 0.3, 0.4 and 0.5 (at 0.3, 1.48 against 1.38 s, 166 and 185 trials,
KS p = 0.067). Without the shift, keeping only the files that agree at the same index, p is 0.0206
at 0.2, 0.158 at 0.3 (1.40 against 1.28 s), 0.0211 at 0.4 and 0.0022 at 0.5 (17 files, 140 and 115
trials); this reading drops the files whose label trials start at a later tone, which the shift
places. From the hand-scored labels, which need no alignment, p = 0.206 over all win trials and p =
0.16 over the trials of the main test; over mouse means p = 0.055. Subordinates are slower in every
reading. With each file's tones placed by its best shift, the difference is significant only when
the files whose labels and entries disagree are kept (cut-offs 0 and 0.1). The outcome takes the
reading with the best shift and the files at r of at least 0.3 (an assumption), and it rests on that
choice: the files aligned without a shift give p below 0.05 at every cut-off except 0.3.

### ed5e-port-occupancy-not-different

> Percent port occupancy during the tone did not differ by relative rank in the recording cohort.
> (paired t-test p = 0.90, 31 sessions)

Percent of the 10 s tone (label bins 0 to 10 s) labelled "self in port" (4, 5) and "other in port"
(6, 7), dom against sub, per session (main) and per file (a), as in ed5c-rewards-not-different.

In [ ]:
tone = slice(int(-c.LABEL_START_S * c.LABEL_HZ), int((c.TONE_S - c.LABEL_START_S) * c.LABEL_HZ))
occ = per_file(lambda r: (100 * np.isin(r["labels"][:, tone], c.SELF_IN).mean(), 100 * np.isin(r["labels"][:, tone], c.OTHER_IN).mean()))
paired(per_session(occ), "(main) sessions")
paired(occ, "(a) files")
print("Range of percent occupancy:", round(min(occ["dom"].min(), occ["sub"].min()), 1), "to", round(max(occ["dom"].max(), occ["sub"].max()), 1))

**Result: reproduced.** Port occupancy during the tone did not differ by relative rank: dominants
26.37% and subordinates 20.23% over 20 sessions (main; paired t-test p = 0.284, mean difference 6.14
points, 95% CI -4.79 to 17.08), and 25.19% against 20.72% over 34 files (p = 0.327).

### ed5f-dominants-displace-more

> Relative dominants were more successful at displacing subordinates from the port, over the whole
> competition and during the tone. (paired t-tests, p = 0.002, 32 sessions; p = 0.005, 31 sessions)

Pushing success is the percent of a mouse's pushes that displaced the competitor (Methods). A push
is a run of bins labelled 1 in the pusher's file; it succeeded when the competitor's file labels any
of its bins 3. This needs the files of both mice, so the test runs over the sessions that have both.
"Whole competition" is all labelled bins (-2 to +12 s of each trial), as the labels cover only the
trials; "tone" is 0 to 10 s.

In [ ]:
pairs = [v for v in lab_by_session.values() if len(v) == 2]
def success(n, s):
    return 100 * s / n if n else np.nan
rows = []
for v in pairs:
    d0, s0 = (v[0], v[1]) if v[0]["rank"] < v[0]["other_rank"] else (v[1], v[0])
    n_t = min(d0["labels"].shape[0], s0["labels"].shape[0])
    D, S = d0["labels"][:n_t], s0["labels"][:n_t]
    row = {"session": "-".join(c.session_key(d0)), "dom_mouse": d0["mouse"], "sub_mouse": s0["mouse"]}
    for name, b in (("whole", slice(None)), ("tone", tone)):
        nd, sd = c.push_success(D, S, b); ns, ss = c.push_success(S, D, b)
        row[f"{name}_dom_pushes"], row[f"{name}_dom"] = nd, success(nd, sd)
        row[f"{name}_sub_pushes"], row[f"{name}_sub"] = ns, success(ns, ss)
    rows.append(row)
ps = pd.DataFrame(rows)
print(ps.round(1).to_string(index=False))
for name in ("whole", "tone"):
    paired(ps.rename(columns={f"{name}_dom": "dom", f"{name}_sub": "sub"}), name)

**Result: reproduced.** Over the 13 sessions with both files and pushes by both mice, dominants
displaced the competitor in 91.7% of their pushes and subordinates in 60.2% over the labelled bins
(paired t-test p = 0.0051), and 93.4% against 56.5% during the tone (p = 0.0013).

### ed5g-time-displaced-not-different

> Total time being displaced from the port did not differ by relative rank in the recording cohort.
> (paired t-test p = 0.15, 31 sessions)

Seconds labelled 3 (displaced) in each mouse's own file, over the labelled bins, dom against sub,
over the sessions with both files, and during the tone only as a variant. A second variant compares
the files of dominants and subordinates unpaired, over all labelled files.

In [ ]:
rows = []
for v in pairs:
    d0, s0 = (v[0], v[1]) if v[0]["rank"] < v[0]["other_rank"] else (v[1], v[0])
    rows.append({"session": "-".join(c.session_key(d0)), "dom": (d0["labels"] == 3).sum() / c.LABEL_HZ, "sub": (s0["labels"] == 3).sum() / c.LABEL_HZ,
                 "dom_tone": (d0["labels"][:, tone] == 3).sum() / c.LABEL_HZ, "sub_tone": (s0["labels"][:, tone] == 3).sum() / c.LABEL_HZ})
tdp = pd.DataFrame(rows)
print(tdp.round(2).to_string(index=False))
paired(tdp, "time displaced (s), own file")
paired(tdp.drop(columns=["dom", "sub"]).rename(columns={"dom_tone": "dom", "sub_tone": "sub"}), "time displaced during the tone (s)")
one = per_file(lambda r: ((r["labels"] == 3).sum() / c.LABEL_HZ, np.nan))
d = one.assign(t=np.where(one.dom_file, one["dom"], one["sub"]))
a, b = d[d.dom_file].t, d[~d.dom_file].t
print(f"Unpaired variant over all {len(d)} labelled files: dom n = {len(a)} mean {a.mean():.2f} s, sub n = {len(b)} mean {b.mean():.2f} s, Welch p = {stats.ttest_ind(a, b, equal_var=False).pvalue:.3g}")

**Result: reproduced.** Time displaced did not differ by relative rank over the 14 sessions with both
files: dominants 3.21 s and subordinates 3.93 s over the labelled bins (paired t-test p = 0.359), and
2.71 against 3.14 s during the tone (p = 0.512).

## HMM-GLM decoding of behaviour from mPFC multi-unit activity

### Input (Methods 17)

> "Multiunit firing rate activity was determined by applying a common reference average and
> determining spikes using 4 standard deviations from the mean. [...] The dataset consists of 965
> trials, with spiking information from 32 channels across 70 time stamps of 200 ms intervals."
> (Methods 17, Dataset)

The input is computed here from the raw `ElectricalSeries` of the labelled files: 600 to 6,000 Hz
band (Methods 11), common average reference, crossings of 4 SD from the channel mean (reading a,
negative crossings below mean minus 4 SD; reading b, crossings in either direction), counted in
70 bins of 200 ms from -2 s to +12 s around tone onset (`competition.mua_trials`). Labels are the
files' 20 Hz labels taken every fourth bin (labels change only on whole seconds, so this is exact).

The recording does not share the behaviour clock, and the offset is in no file (data check). A
file is used when the offset can be placed: a clear port-entry offset (more than twice the 99th
percentile over offsets), an offset within 10 s of the median of the other clear files recorded
that day in that cage, scaled spikes matching the recording (peak above 10 SD), and labels matching
the port entries (r of at least 0.3 at the file's best tone shift k). The window of trial *j* starts
in the recording at tone *j + k* + offset to the spikes - offset of the spikes to the recording - 2 s.
As checks, the multi-unit counts are correlated with the scaled spikes in the same bins (a check of
the recording's offset to the spikes), and both are compared with the paper's own processed input
(`dataset_forHMMGLM_proc.mat` in the code repository at the pinned commit, a variant input that
the claims file names), whose trials are matched to the files' trials by their labels.

In [ ]:
hm = pe.merge(shift[["file", "best_k", "r_best", "label_trials"]], on="file")
hm["group"] = hm.file.str[:8] + "-" + hm.file.str.extract(r"-([A-Za-z]+\d)-")[0]
hm["group_dev"] = (hm.pe_lag - hm.group.map(hm[hm.clear].groupby("group").pe_lag.median())).abs()
hm["use"] = hm.clear & (hm["raw_peak_sd_1.5"] > 10) & (hm.group_dev < 10) & (hm.r_best >= 0.3)
hm["raw_offset"] = hm.pe_lag - hm["raw_lag_1.5"]
hm = hm.sort_values("file").reset_index(drop=True)
print(hm[["file", "label_trials", "clear", "pe_lag", "group_dev", "raw_peak_sd_1.5", "raw_lag_1.5", "best_k", "r_best", "use"]].round(2).to_string(index=False))
byname = {r["path"].split("/")[0][4:]: r for r in lf}
amap = {a["path"]: a for a in al}

def build(row):
    r = byname[row.file]
    k = int(row.best_k)
    on = r["cs"][k:k + r["labels"].shape[0]]
    out = c.mua_trials(amap[r["path"]], on + row.raw_offset)
    sp = np.concatenate(c.spike_units(c.open_nwb(amap[r["path"]]["url"])))
    out["spikes"] = np.array([np.histogram(sp, t + row.pe_lag + c.LABEL_START_S + np.arange(c.MUA_N_BINS + 1) * c.MUA_BIN_S)[0] for t in on])
    out["spike_times"] = sp
    out["onsets"] = on
    return out

started = time.time()
with ThreadPoolExecutor(4) as ex:
    mua = list(ex.map(build, [row for _, row in hm[hm.use].iterrows()]))
print(f"\nmulti-unit counts of {len(mua)} files in {time.time() - started:.0f} s")
used = hm[hm.use].reset_index(drop=True)
meta = pd.concat([pd.DataFrame({"file": row.file, "mouse": byname[row.file]["mouse"],
                                "dom": byname[row.file]["rank"] < byname[row.file]["other_rank"],
                                "trial": np.arange(byname[row.file]["labels"].shape[0])}) for _, row in used.iterrows()], ignore_index=True)
Xn = np.concatenate([m["neg"] for m in mua]); Xb = np.concatenate([m["both"] for m in mua])
Yh = np.concatenate([byname[f]["labels"][:, ::4] - 1 for f in used.file])
keep = ~np.isnan(Xn).any(axis=(1, 2))
print("Files used:", len(used), "of", len(hm), "labelled files with port entries; trials:", len(Xn), "; windows outside the recording (dropped):", int((~keep).sum()))
Xn, Xb, Yh, meta = Xn[keep], Xb[keep], Yh[keep], meta[keep].reset_index(drop=True)
print("Mice:", meta.mouse.nunique(), "; trials in files of relative dominants:", int(meta.dom.sum()), "; of relative subordinates:", int((~meta.dom).sum()))
print("Input shape:", Xn.shape, "; labels per trial:", Yh.shape[1])
for name, X in (("(a) negative", Xn), ("(b) either direction", Xb)):
    rate = X.mean(axis=(0, 2)) / c.MUA_BIN_S
    print(f"Multi-unit rate per channel, {name} (Hz): median {np.median(rate):.2f}, range {rate.min():.2f} to {rate.max():.2f}; counts per bin 0 to {int(X.max())}")
print("Label bins per code:", dict(zip(*np.unique(Yh + 1, return_counts=True))))

In [ ]:
# Check 1: the multi-unit counts (summed over channels) against the scaled spikes of the same file
# in the same bins, per file (median over trials of the correlation of the 70-bin time courses).
rows = []
for row, m in zip(used.itertuples(), mua):
    x = m["neg"].sum(1)
    rr = [np.corrcoef(a, b)[0, 1] for a, b in zip(x, m["spikes"]) if np.std(a) > 0 and np.std(b) > 0]
    sh = [np.corrcoef(a, b)[0, 1] for a, b in zip(x[1:], m["spikes"][:-1]) if np.std(a) > 0 and np.std(b) > 0]
    rows.append({"file": row.file, "r_spikes": np.median(rr), "r_spikes_next_trial": np.median(sh)})
chk = pd.DataFrame(rows)

# Check 2: the paper's processed input. Each file's trial is matched to the paper's trial whose 70
# labels agree best; then the paper's counts are compared with this input, and the spikes of every
# labelled file are scanned over offsets against the paper's counts.
P = hg.paper_input()
PL, PX = P["labels"].astype(int), P["data"].astype(float)
print("Paper's input:", PX.shape, "; animals:", len(np.unique(P["trial2animal"])), "; label bins per code:", dict(zip(*np.unique(PL, return_counts=True))))
def match(lab):
    ag = (lab[:, None, :] == PL[None, :, :]).mean(2)
    s = np.sort(ag, 1)
    return ag.argmax(1), s[:, -1], s[:, -2]
best_i, best_a, second_a = match(Yh + 1)
print(f"Label agreement of each trial with its best match in the paper's input: median {np.median(best_a):.3f} (second best {np.median(second_a):.3f}); trials matched at 0.9 or more: {int((best_a >= 0.9).sum())} of {len(best_a)}")
for col, X in (("r_paper_neg", Xn), ("r_paper_both", Xb)):
    v = [np.corrcoef(X[i].sum(0), PX[best_i[i]].sum(0))[0, 1] if PX[best_i[i]].sum() > 0 and X[i].sum(0).std() > 0 else np.nan for i in range(len(X))]
    chk[col] = pd.Series(v).groupby(meta.file).median().reindex(chk.file).values
    vc = [np.corrcoef(X[i].ravel(), PX[best_i[i]].ravel())[0, 1] if PX[best_i[i]].sum() > 0 else np.nan for i in range(len(X))]
    chk[col + "_by_channel"] = pd.Series(vc).groupby(meta.file).median().reindex(chk.file).values
print(chk.round(2).to_string(index=False))
# Which threshold reading is nearer the paper's counts: total counts of the paper's label-matched trial
# over this input's, on trials matched at 0.9 or more.
good = best_a >= 0.9
print("Paper's input, multi-unit rate per channel (Hz): median", round(float(np.median(PX.mean(axis=(0, 2)) / c.MUA_BIN_S)), 2))
for name, X in (("(a) negative", Xn), ("(b) either direction", Xb)):
    ratio = pd.Series(PX[best_i[good]].sum(axis=(1, 2)) / np.maximum(X[good].sum(axis=(1, 2)), 1)).groupby(meta.file[good].values).median()
    print(f"Paper's counts over this input's, {name}: median over {len(ratio)} files {ratio.median():.2f} (range {ratio.min():.2f} to {ratio.max():.2f})")

def paper_lag(r):
    a = amap[r["path"]]
    sp = np.concatenate(c.spike_units(c.open_nwb(a["url"])))
    rows_ = []
    for k in range(len(r["cs"]) - r["labels"].shape[0] + 1):
        bi, ba, _ = match(r["labels"][:, ::4])
        lags, sc = c.template_lag_scan(sp, r["cs"][k:k + r["labels"].shape[0]], PX[bi].sum(1))
        j = int(np.nanargmax(sc))
        rows_.append({"file": r["path"].split("/")[0][4:], "k": k, "paper_lag": lags[j], "paper_r": sc[j], "paper_p99": np.nanpercentile(sc, 99)})
    return max(rows_, key=lambda x: x["paper_r"])
with ThreadPoolExecutor(8) as ex:
    pl_ = pd.DataFrame(list(ex.map(paper_lag, lf)))
pl_ = pl_.merge(pe[["file", "pe_lag", "clear"]], on="file", how="left").merge(shift[["file", "best_k"]], on="file", how="left")
pl_["clear_paper"] = pl_.paper_r > 2 * pl_.paper_p99
pl_["used"] = pl_.file.isin(used.file)
print(pl_.round(2).to_string(index=False))
both_ = pl_[pl_.clear & pl_.clear_paper]
print("\nLabelled files with a clear peak against the paper's counts:", int(pl_.clear_paper.sum()), "of", len(pl_),
      "; of the files with a clear port-entry offset, the two offsets agree within 1 s in", int(((both_.paper_lag - both_.pe_lag).abs() <= 1).sum()), "of", len(both_))
u = pl_[pl_.used]
print("Files used here: offsets agree within 1 s in", int(((u.paper_lag - u.pe_lag).abs() <= 1).sum()), "of", len(u), "; same tone shift k in", int((u.k == u.best_k).sum()))

The multi-unit counts follow the scaled spikes of the same file in the same bins (median
correlation of the 70-bin time courses 0.40 to 0.96 per file, against about 0 with the next trial's
spikes), so the recording's offset to the spikes is placed correctly in every file used. The
paper's processed input confirms the offset of the spikes to the behaviour clock: scanning the
spikes of each labelled file over offsets against the paper's counts of its label-matched trials
gives a clear peak in all 34 labelled files, at the port-entry offset (within 1 s) in 16 of the 17
files used and with the same tone shift k in all 17. The exception, 20191029-chrimson2-2-2v3, peaks
at -15.8 s against its port-entry offset of -23.7 s; -15.8 s is also the paper's offset for the other
file of that session, so its 30 trials are most likely misaligned here. The paper's input holds 14
animals (Methods 17 says 13), and its rank of each trial gives 487 dom and 478 sub trials, the
counts of the 34 labelled files. The counts computed here correlate less well with the paper's
counts (median per file -0.09 to 0.73 summed over channels, about 0 channel by channel), so the
paper's thresholding or channel order differs from the one used here in ways Methods 17 does not
fix; the paper's input is run as a variant below. On the trials matched at 0.9 or more, the paper's
counts are 1.47 times the negative crossings here (median over the 17 files, range 1.21 to 2.11)
and 1.06 times the crossings in either direction (0.63 to 1.41); the paper's median rate per
channel is 28.15 Hz, against 16.41 and 23.05 Hz here. Crossings in either direction are the nearer
of the two readings to the paper's counts.

### fig1i-hmmglm-decodes-behaviour

> A 6-state HMM-GLM decodes the 9 behaviour labels from mPFC multi-unit activity above chance and
> better than a GLM or an ensemble of 9 GLMs, while the GLM and the 9 GLMs do not differ. (Figure
> 1i; sign test p = 0.004; rank-sum p = 4.0e-5 against each static model; GLM against 9 GLMs p = 0.54;
> expected 965 trials, 32 sessions, 13 mice.)

The HMM-GLM is the code's (`hmmglm.py`, ported from `model_HMMGLM.m` and `run_model.m` at the
pinned commit): 6 states, 100 epochs, 10 folds. The GLM and the 9 GLMs follow Methods 17. Per
behaviour, the AUC is the mean over the 10 folds (folds where a label is absent give no AUC and are
left out of its mean, a standard choice). Main reading, as the criterion lists them first:
negative crossings, counts per bin, AUC with the one-bin shift (`auc_ROC_delay`), contiguous folds
in session order. Each other reading is run with the rest at the main reading. The frequency table
(FT) is shown as the chance reference (AUC 0.5 by construction). The SVM of Methods 17 is not run:
the criterion does not use it, and it enters only the Kruskal-Wallis test across five models,
which the criterion does not judge. The same code is run on the paper's processed input as a
variant, which checks the port against Figure 1i. Two more variants: the main reading without
20191029-chrimson2-2-2v3, the file whose spikes the paper's input places at another offset, and
z-scored rates at learning rates of 1e-6 (the code's), 1e-5 and 1e-4, on this input and on the
paper's.

In [ ]:
def zscore(X):
    m, s = X.mean(axis=(0, 2), keepdims=True), X.std(axis=(0, 2), keepdims=True)
    return (X - m) / np.where(s > 0, s, 1)

def judge(res, key, label):
    A = {k: np.nanmean(v, axis=0) for k, v in res[key].items()}
    h_, g_, n_ = A["HMM-GLM"], A["GLM"], A["9 GLMs"]
    sign = stats.binomtest(int((h_ > 0.5).sum()), 9).pvalue
    rs = lambda a, b: stats.ranksums(a, b).pvalue
    out = {"reading": label, "auc": key, "HMM-GLM min": h_.min(), "HMM-GLM median": np.median(h_), "GLM median": np.median(g_),
           "9 GLMs median": np.median(n_), "HMM > 0.5": int((h_ > 0.5).sum()), "sign p": sign,
           "HMM vs GLM p": rs(h_, g_), "HMM vs 9 GLMs p": rs(h_, n_), "GLM vs 9 GLMs p": rs(g_, n_),
           "HMM > GLM, all 9": bool((h_ > g_).all()), "HMM > 9 GLMs, all 9": bool((h_ > n_).all())}
    out["criterion met"] = bool(sign < 0.05 and out["HMM vs GLM p"] < 0.05 and out["HMM vs 9 GLMs p"] < 0.05 and np.median(h_) > max(np.median(g_), np.median(n_)) and out["GLM vs 9 GLMs p"] >= 0.05)
    return out, A

print("Labelled files (expected values): trials", sum(r["labels"].shape[0] for r in lf), "; files", len(lf), "; mice", len({r["mouse"] for r in lf}),
      "| used here: trials", len(Xn), "; files", meta.file.nunique(), "; mice", meta.mouse.nunique())
readings = {"main": dict(X=Xn), "(b) crossings in either direction": dict(X=Xb), "(b) z-scored per channel": dict(X=zscore(Xn)),
            "(b) random folds": dict(X=Xn, contiguous=False)}
started = time.time()
cv = {}
for i, (name, kw) in enumerate(readings.items()):
    X = kw.pop("X")
    cv[name] = hg.cross_validate(X, Yh, 9, seed=100 + i, **kw)
print(f"fits in {time.time() - started:.0f} s")
rows, aucs = [], {}
for name, res in cv.items():
    for key in ("auc_delay", "auc"):
        o, A = judge(res, key, name)
        rows.append(o); aucs[(name, key)] = A
tab = pd.DataFrame(rows)
print(tab.round(4).to_string(index=False))
print("\nPer-behaviour AUC, main reading, with the shift (rows: models; columns: labels 1 to 9):")
print(pd.DataFrame(aucs[("main", "auc_delay")], index=[c.LABELS[j] for j in range(1, 10)]).T.round(3).to_string())
print("\nFolds per label with no AUC (label absent from the held-out trials), main reading:", dict(zip(range(1, 10), np.isnan(cv["main"]["auc_delay"]["HMM-GLM"]).sum(0))))
print("Range of all per-behaviour AUCs shown:", round(min(np.nanmin(v) for A in aucs.values() for v in A.values()), 3), "to", round(max(np.nanmax(v) for A in aucs.values() for v in A.values()), 3))

In [ ]:
# Variants: the paper's processed input (all 965 trials, in the order of the file, as run_model.m
# takes them), and leave-one-mouse-out for the HMM-GLM (Methods 17, Extended Data Fig. 5j-k), here on
# this input.
started = time.time()
cv_paper = hg.cross_validate(PX, PL - 1, 9, seed=200)
rows = [judge(cv_paper, key, "paper's input")[0] for key in ("auc_delay", "auc")]
print(f"paper's input: {len(PX)} trials, fits in {time.time() - started:.0f} s")
print(pd.DataFrame(rows).round(5).to_string(index=False))
print("Per-behaviour AUC, paper's input, with the shift:")
print(pd.DataFrame(judge(cv_paper, "auc_delay", "")[1], index=[c.LABELS[j] for j in range(1, 10)]).T.round(3).to_string())
mice = sorted(meta.mouse.unique())
loo = hg.cross_validate(Xn, Yh, 9, seed=300, static=False, test_sets=[np.flatnonzero(meta.mouse == m) for m in mice])
L = np.nanmean(loo["auc_delay"]["HMM-GLM"], axis=0)
print(f"\nLeave-one-mouse-out ({len(mice)} mice), HMM-GLM AUC per behaviour with the shift:", L.round(3),
      "; above 0.5:", int((L > 0.5).sum()), "of 9; sign test p =", round(stats.binomtest(int((L > 0.5).sum()), 9).pvalue, 4))
print("Per mouse, median AUC over behaviours:", dict(zip(mice, np.round(np.nanmedian(loo["auc_delay"]["HMM-GLM"], axis=1), 3))))

# Variant: without 20191029-chrimson2-2-2v3, whose spikes the paper's input places at another offset
# (data check above), all models, main reading.
bad = meta.file != "20191029-chrimson2-2-2v3"
cv_drop = hg.cross_validate(Xn[bad.values], Yh[bad.values], 9, seed=310)
print(f"\nWithout 20191029-chrimson2-2-2v3 ({int(bad.sum())} trials, {meta[bad].file.nunique()} files, {meta[bad].mouse.nunique()} mice):")
print(pd.DataFrame([judge(cv_drop, key, "without 2-2-2v3")[0] for key in ("auc_delay", "auc")]).round(4).to_string(index=False))

# Variant: rates z-scored per channel with other learning rates than the code's 1e-6, on this input and
# on the paper's input (HMM-GLM only; the static fits scale each channel by its SD, so z-scoring
# changes them only by centring).
zrows = []
for name, X, Y in (("this input", zscore(Xn), Yh), ("paper's input", zscore(PX), PL - 1)):
    for i, eta in enumerate((1e-6, 1e-5, 1e-4)):
        res = hg.cross_validate(X, Y, 9, seed=320 + i + (10 if name != "this input" else 0), static=False, eta0=eta)
        for key in ("auc_delay", "auc"):
            A = np.nanmean(res[key]["HMM-GLM"], axis=0)
            zrows.append({"input (z-scored)": name, "learning rate": eta, "auc": key, "HMM > 0.5": int((A > 0.5).sum()),
                          "min": A.min(), "median": np.median(A), "max": A.max(), "sign p": stats.binomtest(int((A > 0.5).sum()), 9).pvalue,
                          "held-out log-lik": np.mean(res["loglik"])})
print("\nZ-scored rates, HMM-GLM AUC per behaviour (median over the 9 labels) by learning rate:")
print(pd.DataFrame(zrows).round(4).to_string(index=False))

**Result: reproduced, on 17 of the 34 labelled files.** On the 512 trials (17 files, 10 mice) whose
recording can be placed on the behaviour clock, the 6-state HMM-GLM decodes all 9 labels above
chance (AUC 0.60 to 0.83 with the one-bin shift, median 0.774; sign test p = 0.0039), above the GLM
(median 0.541; rank-sum p = 0.0003) and the 9 GLMs (median 0.539; p = 0.0003), while the GLM and
the 9 GLMs do not differ (p = 0.627). Without the shift: HMM-GLM median 0.713, p = 0.0009 against
each static model. Crossings in either direction give the same outcome (HMM-GLM median 0.910). With
random folds the HMM-GLM still beats both static models with the shift (p = 0.047) but not without
it (p = 0.233), as the static models gain from random folds (median 0.63). Without
20191029-chrimson2-2-2v3 (482 trials, 16 files, 9 mice) every test passes as before (HMM-GLM median
0.693, min 0.637; p = 0.0003 against each static model; GLM against 9 GLMs p = 0.627). With rates
z-scored per channel the HMM-GLM fails: median 0.475 and 0 of 9 above 0.5 in the table of readings,
and median 0.511 with 7 of 9 above 0.5 (sign p = 0.18) at the same learning rate of 1e-6 and another
seed, so at the code's rate the AUCs lie near 0.5, on either side of it with the seed. Z-scored
rates of the paper's own input also give chance at that rate (median 0.494, 4 of 9 above 0.5), and
at 1e-5 (medians 0.484 here and 0.465 on the paper's input). At 1e-4 the HMM-GLM decodes all 9
labels from z-scored rates, here (median 0.576, sign p = 0.0039) and on the paper's input (median
0.650). The code's learning rate is thus set for counts, which the paper's input holds, and z-scored
rates are not the paper's reading. The outcome rests on the readings the code and the paper's input
support (counts, contiguous folds), an assumption. Leave-one-mouse-out gives AUCs above 0.5 for all
9 labels (sign test p = 0.0039). On the paper's own processed input the ported code gives HMM-GLM
AUCs of 0.79 to 0.98 against static medians near 0.59 (every test as the criterion asks), which
matches Figure 1i. The labelled files hold 965 trials, 34 files and 14 mice, within the expected
965, 32 and 13; the 512 trials, 17 files and 10 mice analysed are outside those tolerances.

### ed5j-six-states

> Held-out log-likelihood of the HMM-GLM increases with the number of hidden states and plateaus at
> 6 states. (Extended Data Fig. 5j)

Per number of states, 1 to 10, the held-out log-likelihood averaged over the trials of each fold,
with its SEM over the 10 folds (main reading of fig1i-hmmglm-decodes-behaviour). Readings: (a) the
smallest number whose mean lies within one SEM of the best mean, (b) the number with the highest
mean. Also run with crossings in either direction, and on the paper's processed input as a
variant.

In [ ]:
started = time.time()
ll = {}
for k in range(1, 11):
    ll[k] = np.array(hg.cross_validate(Xn, Yh, 9, seed=400 + k, k_states=k, static=False)["loglik"])
print(f"fits in {time.time() - started:.0f} s")
llt = pd.DataFrame({"states": list(ll), "mean": [v.mean() for v in ll.values()], "sem": [v.std(ddof=1) / np.sqrt(len(v)) for v in ll.values()],
                    "min over folds": [v.min() for v in ll.values()], "max over folds": [v.max() for v in ll.values()]})
print(llt.round(3).to_string(index=False))
best = int(llt.loc[llt["mean"].idxmax(), "states"])
within = llt[llt["mean"] >= llt["mean"].max() - llt.loc[llt["mean"].idxmax(), "sem"]]
print("(a) smallest number within one SEM of the best:", int(within.states.min()), "; (b) highest mean:", best)
d = np.array([ll[k] - ll[1] for k in ll])
print("Gain over 1 state, mean over folds:", dict(zip(ll, d.mean(1).round(3))))
# Reading (b) of the threshold, crossings in either direction.
started = time.time()
llb = {k: np.array(hg.cross_validate(Xb, Yh, 9, seed=450 + k, k_states=k, static=False)["loglik"]) for k in range(1, 11)}
print(f"\nCrossings in either direction, 1 to 10 states, fits in {time.time() - started:.0f} s")
llbt = pd.DataFrame({"states": list(llb), "mean": [v.mean() for v in llb.values()], "sem": [v.std(ddof=1) / np.sqrt(len(v)) for v in llb.values()]})
print(llbt.round(3).to_string(index=False))
wb = llbt[llbt["mean"] >= llbt["mean"].max() - llbt.loc[llbt["mean"].idxmax(), "sem"]]
print("Either direction: (a) smallest number within one SEM of the best:", int(wb.states.min()), "; (b) highest mean:", int(llbt.loc[llbt["mean"].idxmax(), "states"]))
started = time.time()
llp = {k: np.array(hg.cross_validate(PX, PL - 1, 9, seed=500 + k, k_states=k, static=False)["loglik"]) for k in range(1, 11)}
print(f"\nPaper's input (variant), 1 to 10 states, fits in {time.time() - started:.0f} s")
llpt = pd.DataFrame({"states": list(llp), "mean": [v.mean() for v in llp.values()], "sem": [v.std(ddof=1) / np.sqrt(len(v)) for v in llp.values()],
                     "min over folds": [v.min() for v in llp.values()], "max over folds": [v.max() for v in llp.values()]})
print(llpt.round(3).to_string(index=False))
bp = int(llpt.loc[llpt["mean"].idxmax(), "states"])
wp = llpt[llpt["mean"] >= llpt["mean"].max() - llpt.loc[llpt["mean"].idxmax(), "sem"]]
print("Paper's input: (a) smallest number within one SEM of the best:", int(wp.states.min()), "; (b) highest mean:", bp)

**Result: not reproduced.** The held-out log-likelihood rises from 1 to 2 states (-154.8 to -124.5
per trial) and then falls slowly to -126.9 at 10 states; both readings select 2 states (6 states
give -126.1, SEM 4.3). With crossings in either direction it is highest at 4 states (-121.3, SEM
3.8) and the smallest number within one SEM of that is 3, so no reading selects 5, 6 or 7. On the
paper's input the same code rises from -529.6 at 1 state to -126.5 at 6 states (SEM 0.57) and is
lower at every larger number (-127.6 at 10), and both readings select 6, as the paper reports. The
port selects 6 states on the paper's input and not on the input computed from the dandiset, so the
difference comes from the input, not from the code.

### ed5k-hmmglm-generalizes-across-rank

> The 6-state HMM-GLM predicts behaviour above chance whichever relative rank group is used for
> training and for testing. (Extended Data Fig. 5k; sign test p = 0.004; 482 dom and 478 sub trials.)

Trials of files whose subject is the relative dominant (dom) or subordinate (sub). The model of
fig1i-hmmglm-decodes-behaviour (main reading) trained on all dom trials and tested on all sub trials,
the reverse, and 10-fold runs within each group. Per behaviour the AUC on the test trials, with the
shift (main) and without. The readings of fig1i-hmmglm-decodes-behaviour that also met its
criterion are run too: crossings in either direction (all four combinations) and random folds (the
within-group runs; the cross-group runs have no folds). On the paper's input, both directions and
both within-group runs.

In [ ]:
dom_i, sub_i = np.flatnonzero(meta.dom), np.flatnonzero(~meta.dom)
print("Trials: dom", len(dom_i), "(files", meta[meta.dom].file.nunique(), ", mice", meta[meta.dom].mouse.nunique(), "); sub", len(sub_i),
      "(files", meta[~meta.dom].file.nunique(), ", mice", meta[~meta.dom].mouse.nunique(), ")")
rows = []
for name, (tr, te) in {"dom to sub": (dom_i, sub_i), "sub to dom": (sub_i, dom_i)}.items():
    a0, a1 = hg.train_test(Xn[tr], Yh[tr], Xn[te], Yh[te], 9, seed=600 + len(rows))
    rows.append((name, a1, a0))
for name, idx in {"dom 10-fold": dom_i, "sub 10-fold": sub_i}.items():
    res = hg.cross_validate(Xn[idx], Yh[idx], 9, seed=700 + len(rows), static=False)
    rows.append((name, np.nanmean(res["auc_delay"]["HMM-GLM"], 0), np.nanmean(res["auc"]["HMM-GLM"], 0)))
out = []
for name, a1, a0 in rows:
    for lab_, a in (("with shift", a1), ("without shift", a0)):
        ok = ~np.isnan(a)
        out.append({"train/test": name, "auc": lab_, "labels with an AUC": int(ok.sum()), "above 0.5": int((a[ok] > 0.5).sum()),
                    "min": np.nanmin(a), "median": np.nanmedian(a), "sign p": stats.binomtest(int((a[ok] > 0.5).sum()), int(ok.sum())).pvalue,
                    "per label": " ".join(f"{x:.2f}" for x in a)})
ek = pd.DataFrame(out)
print(ek.round(4).to_string(index=False))
print("\nEvery combination at sign p < 0.05, with shift:", bool((ek[ek.auc == "with shift"]["sign p"] < 0.05).all()),
      "; without shift:", bool((ek[ek.auc == "without shift"]["sign p"] < 0.05).all()))
# Readings of fig1i-hmmglm-decodes-behaviour that also met its criterion: crossings in either
# direction (all four combinations), and random folds (the within-group runs; the cross-group runs
# have no folds).
rows_b = []
for name, (tr, te) in {"dom to sub": (dom_i, sub_i), "sub to dom": (sub_i, dom_i)}.items():
    a0, a1 = hg.train_test(Xb[tr], Yh[tr], Xb[te], Yh[te], 9, seed=900 + len(rows_b))
    rows_b.append(("(b) either direction", name, a1, a0))
for name, idx in {"dom 10-fold": dom_i, "sub 10-fold": sub_i}.items():
    res = hg.cross_validate(Xb[idx], Yh[idx], 9, seed=910 + len(rows_b), static=False)
    rows_b.append(("(b) either direction", name, np.nanmean(res["auc_delay"]["HMM-GLM"], 0), np.nanmean(res["auc"]["HMM-GLM"], 0)))
    res = hg.cross_validate(Xn[idx], Yh[idx], 9, seed=920 + len(rows_b), static=False, contiguous=False)
    rows_b.append(("(b) random folds", name, np.nanmean(res["auc_delay"]["HMM-GLM"], 0), np.nanmean(res["auc"]["HMM-GLM"], 0)))
out_b = []
for reading, name, a1, a0 in rows_b:
    for lab_, a in (("with shift", a1), ("without shift", a0)):
        ok = ~np.isnan(a)
        out_b.append({"reading": reading, "train/test": name, "auc": lab_, "above 0.5": int((a[ok] > 0.5).sum()), "of": int(ok.sum()),
                      "min": np.nanmin(a), "median": np.nanmedian(a), "sign p": stats.binomtest(int((a[ok] > 0.5).sum()), int(ok.sum())).pvalue,
                      "per label": " ".join(f"{x:.2f}" for x in a)})
print("\nOther readings:")
print(pd.DataFrame(out_b).round(4).to_string(index=False))
# Variant: the paper's input, with its own rank of each trial (trial2relrank; 487 trials above 0 and
# 478 below, the dom and sub counts of the labelled files here).
pr = P["trial2relrank"].ravel()
pd_i, ps_i = np.flatnonzero(pr > 0), np.flatnonzero(pr < 0)
for name, (tr, te) in {"dom to sub": (pd_i, ps_i), "sub to dom": (ps_i, pd_i)}.items():
    a0, a1 = hg.train_test(PX[tr], PL[tr] - 1, PX[te], PL[te] - 1, 9, seed=800 + len(name))
    print(f"paper's input, {name} ({len(tr)} to {len(te)} trials), with shift: above 0.5 in {int((a1 > 0.5).sum())} of 9, min {a1.min():.3f}, sign p = {stats.binomtest(int((a1 > 0.5).sum()), 9).pvalue:.4f}; per label", " ".join(f"{x:.2f}" for x in a1))
for name, idx in {"dom 10-fold": pd_i, "sub 10-fold": ps_i}.items():
    res = hg.cross_validate(PX[idx], PL[idx] - 1, 9, seed=850 + (name == "sub 10-fold"), static=False)
    a1 = np.nanmean(res["auc_delay"]["HMM-GLM"], 0)
    print(f"paper's input, {name} ({len(idx)} trials), with shift: above 0.5 in {int((a1 > 0.5).sum())} of 9, min {a1.min():.3f}, sign p = {stats.binomtest(int((a1 > 0.5).sum()), 9).pvalue:.4f}; per label", " ".join(f"{x:.2f}" for x in a1))

**Result: reproduced, with crossings in either direction (an assumption).** With crossings in either
direction every combination passes: dom to sub and sub to dom give 9 of 9 labels above 0.5 (medians
0.821 and 0.788, lowest AUC 0.780 and 0.582), and the 10-fold runs within each group 9 of 9 (p =
0.0039), with and without the shift. With negative crossings, trained on dominants' trials and
tested on subordinates', the HMM-GLM is above chance for 8 of 9 labels (sign test p = 0.039), and
within each group by 10-fold cross-validation for 9 of 9 (p = 0.0039); trained on subordinates'
trials (213 trials from 4 mice) and tested on dominants', it is above 0.5 for only 7 of 9 labels (p
= 0.18; AUCs 0.48 to 0.91), with and without the shift, so not every combination passes on that
reading. With random folds the within-group runs also give 9 of 9. The outcome thus depends on the
threshold reading. It takes crossings in either direction, as they are the nearer to the paper's
own counts (1.06 against 1.47 times, data check of the HMM-GLM input); that choice was made after
the result was seen, and the counts here follow the paper's poorly in time under both readings, so
the outcome rests on it as an assumption. Figure 1i and ED Fig. 5j keep negative crossings as their
main reading and have the same outcome under both. On the paper's input (487 dom and 478 sub
trials) both directions and both within-group runs give 9 of 9 above 0.5 (p = 0.0039).

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| ed5b-rank1-wins-most | reproduced |
| ed5c-rewards-not-different | partially reproduced (rewards part holds; no body weights in the dandiset) |
| ed5d-subordinates-slower-latency | not reproduced (with each file's tones placed by its best shift, significant only when files whose labels and port entries disagree are kept; assumption; the files aligned without a shift give p < 0.05 at cut-offs other than 0.3) |
| ed5e-port-occupancy-not-different | reproduced |
| ed5f-dominants-displace-more | reproduced |
| ed5g-time-displaced-not-different | reproduced |
| fig1i-hmmglm-decodes-behaviour | reproduced, on 17 of 34 labelled files (assumption: counts and contiguous folds) |
| ed5j-six-states | not reproduced (held-out likelihood highest at 2 states, or 3 to 4 with crossings in either direction; 6 on the paper's input) |
| ed5k-hmmglm-generalizes-across-rank | reproduced, with crossings in either direction (assumption; with negative crossings, training on subordinates' trials gives 7 of 9 labels above chance) |
| fig1b, fig1c, ed1d, ed1g, fig1e | blocked, data missing (unimplanted cohort, AlphaTracker evaluation frames) |
| ed7b-distance-by-trial-type-not-rank | blocked, metadata insufficient (no port location, no pose identity) |
| 22 single-unit claims of Figures 2 and 3 and ED Figures 6 to 9 | blocked, metadata insufficient (curated cells not marked) |
| ed8ij, fig2e, ed9b | blocked, data missing (alone sessions) |
| fig4a, fig4b-phototagged-counts, fig4b-lh-stronger-reward-excitation, ed10c, fig4d, ed10ek | blocked, data missing |

How the analysis differs from the paper's. The behavioural tests use the hand-scored labels of
the 34 labelled files (and the port entries for latency), not the paper's AlphaTracker distance or its
GUI scoring of whole sessions; the labels cover -2 to +12 s of each trial, so "whole competition"
means the labelled bins. Rewards and port occupancy are compared per session (20), with the 34 files
as a variant. Pushing success and time displaced need both mice's files and use 13 and 14 sessions.
Reward delivery is taken at 2 s, from the Methods; the files give 0.49 s. Latency (ED Fig. 5d) is
judged on the trials whose labels can be placed on the port entries. The HMM-GLM claims use the
paper's code, ported, on multi-unit counts computed here from the raw recording of the 17 labelled
files (512 trials, 10 mice) whose recording can be placed on the behaviour clock, against the
paper's 965 trials and 13 mice; the paper's own processed input is run as a variant, and on it the
port reproduces Figure 1i, selects 6 states for ED Fig. 5j, and passes all four train and test
combinations of ED Fig. 5k. The SVM of Figure 1i is not run.

What was found about the dataset. The stored spike times are two thirds of their time in the
recording (divided by 30 kHz, with a 20 kHz recording). In 18 of the 21 mPFC-LH files and one
"ephys" file the scaled spikes match the recording only after an offset of 5 to 28 s, and in the
other 3 mPFC-LH files at no offset within 60 s. The offset between the neural
recording and the behaviour clock (the `video_gap` the description names) is in no file; the data
put it at 4 to 45 s in 25 of the 26 files where it can be estimated (294 s in the other), differing by
session, and the paper's processed input confirms it in 16 of the 17 files used for the HMM-GLM.
Every cluster is labelled `unsorted` (5,879 against the paper's 998 cells), so the paper's cells
cannot be selected. In 18 labelled files the labels hold fewer trials than tones, and in several of
them the label trials start at a later tone; nothing in the files says which tones were scored.
The "ephys" files have no port entries, labels or poses; the BLA files have poses with every value
missing; no file holds alone sessions, phototagging pulses, the optogenetic experiments or body
weights. The tone schedule is the program's, the same in all files of a cohort, deliveries are 0.49
s after onset against the paper's 2 s, `pair_in_dataset` says "not included" where the competitor's
file is present, one session start time and one subject description contradict the file names.